# Trascrivi Lezioni — Google Colab

Transcribe your lectures (audio/video **files** or **links** like YouTube/Webex)
into cleaned, translated **Word documents**, entirely in the cloud — nothing to
install on your computer. Powered by Google Gemini.

> This is the easiest way to use the project: just run the cells in order.

## 1. Get a free Gemini API key
Create one (free) at **https://aistudio.google.com/apikey** and copy it.

## 2. Store the key as a Colab *Secret* (recommended)
In the left sidebar click the **🔑 key icon** → **＋ Add new secret**:
- **Name:** `GOOGLE_API_KEY`
- **Value:** your key
- turn ON **Notebook access**

This keeps your key hidden and out of the notebook. (If you skip this, the
notebook will simply ask you to paste the key when you run it.)

## 3. Add your lectures
- **Files:** click the **📁 Files** icon in the left sidebar and **upload** your
  video/audio into the `/content` folder (drag & drop). Then list them below.
- **Links:** just paste YouTube/Webex URLs in the list below — no upload needed.
- **Webex** recordings need a `cookies.txt` (exported from your browser while
  logged in): upload it too and set `COOKIES` below.

## 4. Run the cells in order (▶)
Edit the **Settings** cell to list your lectures, then run everything. At the end
you can download all the Word files as a `.zip`.

---
**Privacy:** your audio and transcripts are sent to Google Gemini for processing.
Only transcribe recordings you are allowed to use. A free CPU runtime is enough
(no GPU needed — Gemini does the heavy lifting).


In [ ]:
#@title Step 1 — Install (run once) { display-mode: "form" }
# Installs the transcriber and yt-dlp in this Colab session. ffmpeg is already
# available on Colab. Takes ~1 minute.
!pip install -q "git+https://github.com/GiorgioAlo/lecture-transcriber"
print("Installed. You can move on to the next cell.")


In [ ]:
#@title Step 2 — Load your API key { display-mode: "form" }
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets (GOOGLE_API_KEY).")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key (hidden): ")
    print("API key set for this session.")


In [ ]:
#@title Step 3 — Settings (edit me) { display-mode: "form" }
# List your lectures: uploaded files (paths under /content) and/or links.
SOURCES = [
    "/content/lecture1.mp4",
    # "/content/lecture2.mp4",
    # "https://www.youtube.com/watch?v=XXXXXXXX",
]

LANGUAGES  = ["it", "en"]              # output document languages
OUTPUT_DIR = "/content/transcriptions" # where the Word files are saved
COURSE     = ""                        # optional, e.g. "Computer Security"
COOKIES    = None                      # e.g. "/content/cookies.txt" for Webex links

# Optional: transcribe only a clip (for a quick test). None = whole lecture.
START_MIN   = 0
MINUTES     = None
print(f"{len(SOURCES)} lecture(s) queued -> {OUTPUT_DIR}")


In [ ]:
#@title Step 4 — Transcribe { display-mode: "form" }
import os, shutil
from pathlib import Path
from trascrittore import Job, esegui
from trascrittore.progress import ReporterConsole
from trascrittore import config

key = os.environ.get("GOOGLE_API_KEY", "")
assert key, "No API key — run Step 2 first."

out = Path(OUTPUT_DIR); out.mkdir(parents=True, exist_ok=True)
work = Path("/content/.cache")

def is_url(s): return s.lower().startswith(("http://", "https://"))

saved, failed = [], 0
for i, src in enumerate(SOURCES, 1):
    link = is_url(src)
    name = "Lecture" if link else Path(src).stem
    print(f"
=== [{i}/{len(SOURCES)}] {name} ===")
    if not link and not Path(src).exists():
        print(f"  [x] file not found: {src} (did you upload it to /content?)"); failed += 1; continue
    job = Job(
        sorgente=None if link else Path(src),
        url=src if link else None,
        nome=name, lingue_output=LANGUAGES, corso=COURSE, cookies=COOKIES,
        inizio=START_MIN, minuti=MINUTES,
    )
    try:
        ris = esegui(job, key, ReporterConsole(config.FASI), work)
    except Exception as e:
        print(f"  [x] failed: {e}"); failed += 1; continue
    for d in ris.documenti:
        dest = out / Path(d).name
        shutil.copy2(d, dest); saved.append(dest)
        print("  [doc]", dest)

print(f"
Done. {len(saved)} document(s) in {out}" + (f"  ({failed} failed)" if failed else ""))


In [ ]:
#@title Step 5 — Download all results as a .zip { display-mode: "form" }
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/transcriptions", "zip", OUTPUT_DIR)
files.download(zip_path)
